# B5 · 综合项目：字符级语言模型

把前面四节全部用上，做一个**能续写文本的小模型**。

这件东西的结构，和 GPT 是同一套骨架：给定前面的字符，预测下一个字符。

**学完这一节你应该能**
- 从原始文本构造出训练样本
- 用 one-hot 编码 + 一个小网络完成分类
- 训练、评估、采样一整套流程独立写出来

In [ ]:
import torch


def check(name, ok, hint=""):
    if ok:
        print(f"✓ {name}")
    else:
        print(f"✗ {name}")
        if hint:
            print(f"   提示：{hint}")

## 1. 语料

语料小、词表小，才能在 CPU 上几分钟训出来。

In [ ]:
text = (
    "the quick brown fox jumps over the lazy dog.\n"
    "the dog sleeps and the fox runs away.\n"
    "a quick fox sees a lazy dog and runs.\n"
    "the lazy dog sleeps while the quick fox jumps.\n"
    "over the lazy dog the quick brown fox jumps again.\n"
    "the fox and the dog become friends one day.\n"
)

print("字符数:", len(text))
chars = sorted(set(text))
print("词表大小:", len(chars))
print("词表:", "".join(chars))

In [ ]:
# 字符 <-> 序号
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

print("\n".join(f"{c!r} -> {i}" for c, i in list(stoi.items())[:8]))

## 2. 构造训练样本

最简单的形式：**用 1 个字符预测下 1 个字符**。

把整段文本切成相邻字符对 `(当前字符, 下一个字符)`，就是训练集。

In [ ]:
xs = [stoi[c] for c in text[:-1]]
ys = [stoi[c] for c in text[1:]]

print("样本数:", len(xs))
print("前 10 对:", [(itos[x], itos[y]) for x, y in zip(xs[:10], ys[:10])])

### one-hot 编码

模型只会算数，不会直接吃字符序号。把序号变成**只有一个 1 的向量**：

序号 3、词表大小 30 → 长度 30 的向量，第 3 位是 1，其余是 0。

这是最朴素的「词嵌入」（embedding 的一种特例）。

In [ ]:
import torch
import torch.nn as nn

V = len(chars)

X = torch.zeros(len(xs), V)
X[torch.arange(len(xs)), torch.tensor(xs)] = 1.0   # 第 i 行，第 xs[i] 列置 1
y = torch.tensor(ys)

print("X shape:", tuple(X.shape), "（样本数 x 词表大小）")
print("第 0 个样本的 one-hot:", X[0])
print("对应字符:", itos[xs[0]], "→ 目标是:", itos[ys[0]])

## 3. 模型

一层隐藏层的 MLP。输入是 one-hot，输出是「每个字符是下一个的概率」。

In [ ]:
class CharMLP(nn.Module):
    def __init__(self, vocab, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(vocab, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, vocab)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


model = CharMLP(V)
print(model)
print("参数数:", sum(p.numel() for p in model.parameters()))

logits = model(X[:1])
print("\n一个样本的输出 shape:", tuple(logits.shape), " ← 词表大小个分数，还不是概率")
print("过一遍 softmax 就是概率:", torch.softmax(logits, dim=-1).sum().item())

## 4. 训练

用 `CrossEntropyLoss`。它内部自带 softmax，所以模型输出**原始分数（logits）**就行，不要再手动 softmax。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True, num_workers=0)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(200):
    total = 0.0
    for xb, yb in loader:
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:>3} | loss {total / len(loader):.4f}")

In [ ]:
# 随机猜的 loss 应该是 ln(V)，看现在低了多少
import math

print("随机猜的 loss:", round(math.log(V), 4))
with torch.no_grad():
    final = loss_fn(model(X), y).item()
print("训练后 loss  :", round(final, 4))

## 5. 采样：让它续写

给一个起始字符，反复「预测下一个 → 接上 → 再预测」。

两种取法：

- **贪心**：每次取概率最大的 → 稳定，但可能循环
- **按概率采样**：按概率抽 → 更像人写的

In [ ]:
@torch.no_grad()
def generate(model, start="t", n=200, temperature=1.0, greedy=False):
    model.eval()
    idx = stoi[start]
    out = [idx]

    for _ in range(n):
        x = torch.zeros(1, V)
        x[0, idx] = 1.0
        logits = model(x)[0]

        if greedy:
            idx = int(torch.argmax(logits))
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            idx = int(torch.multinomial(probs, 1))

        out.append(idx)

    model.train()
    return "".join(itos[i] for i in out)


print("贪心（稳定）：")
print(generate(model, "t", 120, greedy=True))
print("\n采样（多样）：")
print(generate(model, "t", 120, temperature=1.0))

In [ ]:
# 温度越低越保守，越高越乱
for T in [0.3, 1.0, 2.0]:
    print(f"--- temperature = {T} ---")
    print(generate(model, "a", 80, temperature=T))
    print()

## 6. 再加一点上下文

只看 1 个字符，模型只能学到「t 后面常跟 h」。看 3 个字符，才能学到「the 后面跟空格」。

做法：把 3 个 one-hot **拼起来**当输入。这正是 transformer 里「上下文窗口」的雏形。

In [ ]:
CTX = 3

xs3, ys3 = [], []
for i in range(len(text) - CTX):
    xs3.append([stoi[c] for c in text[i:i + CTX]])
    ys3.append(stoi[text[i + CTX]])

X3 = torch.zeros(len(xs3), CTX * V)
for row, seq in enumerate(xs3):
    for pos, ch in enumerate(seq):
        X3[row, pos * V + ch] = 1.0
y3 = torch.tensor(ys3)

print("样本数:", len(xs3), " 输入维度:", tuple(X3.shape))

In [ ]:
model3 = CharMLP(CTX * V, hidden=128)
loader3 = DataLoader(TensorDataset(X3, y3), batch_size=64, shuffle=True, num_workers=0)
opt3 = torch.optim.Adam(model3.parameters(), lr=1e-2)

for epoch in range(300):
    total = 0.0
    for xb, yb in loader3:
        loss = loss_fn(model3(xb), yb)
        opt3.zero_grad()
        loss.backward()
        opt3.step()
        total += loss.item()
    if epoch % 60 == 0 or epoch == 299:
        print(f"epoch {epoch:>3} | loss {total / len(loader3):.4f}")

In [ ]:
@torch.no_grad()
def generate3(model, start="the", n=150, temperature=0.8):
    model.eval()
    ctx = [stoi[c] for c in start]
    out = list(ctx)

    for _ in range(n):
        x = torch.zeros(1, CTX * V)
        for pos, ch in enumerate(ctx[-CTX:]):
            x[0, pos * V + ch] = 1.0

        logits = model(x)[0]
        idx = int(torch.multinomial(torch.softmax(logits / temperature, dim=-1), 1))
        out.append(idx)
        ctx.append(idx)

    model.train()
    return "".join(itos[i] for i in out)


print(generate3(model3, "the", 150))

## 7. 练习

In [ ]:
# 练习 1：统计词表大小
V_ans = None                    # ← 改这一行

check("练习 1：词表大小应为 " + str(V), V_ans == V,
      hint="len(set(text))")

In [ ]:
# 练习 2：把字符串 "fox" 编码成序号列表
def encode(s):
    return []                   # ← 改这一行


check("练习 2：'fox' 应编码为 " + str([stoi[c] for c in "fox"]),
      encode("fox") == [stoi[c] for c in "fox"],
      hint="[stoi[c] for c in s]")

In [ ]:
# 练习 3：构造一个 one-hot 张量，让第 2 行第 5 列为 1
onehot = None                   # ← 改这一行

check("练习 3：shape (3, V) 且 [2, 5] == 1",
      onehot is not None
      and tuple(onehot.shape) == (3, V)
      and onehot[2, 5].item() == 1.0
      and onehot.sum().item() == 1.0,
      hint="torch.zeros(3, V) 之后 onehot[2, 5] = 1.0")

In [ ]:
# 练习 4：算出随机猜测时的 loss（交叉熵的理论值）
import math

random_loss = None              # ← 改这一行

check("练习 4：应约等于 ln(V) = " + str(round(math.log(V), 3)),
      random_loss is not None and abs(random_loss - math.log(V)) < 1e-6,
      hint="math.log(V)")

In [ ]:
# 练习 5：用上下文长度 1 的模型，生成 50 个字符（贪心），
# 返回字符串，要求第一个字符是起始字符
def gen_fixed():
    return ""                   # ← 改这一行


s = gen_fixed()
check("练习 5：长度 51 且以 't' 开头",
      isinstance(s, str) and len(s) == 51 and s.startswith("t"),
      hint="generate(model, 't', 50, greedy=True)")

## 收尾

这个模型和 GPT 的差距，只有三处：

1. **上下文**：这里只有 3 个字符，GPT 有几千个 token
2. **结构**：这里是一层全连接，GPT 是几十层 transformer（关键是 attention）
3. **规模**：这里几万个参数，GPT 上千亿

骨架是一样的：**输入上下文 → 输出下一个字符的概率 → 最大化正确字符的概率**。

回答这三个：

1. 为什么 `CrossEntropyLoss` 前面不用手动 softmax？
2. 上下文从 1 个字符加到 3 个，输入维度变成了多少？为什么？
3. 贪心采样和按概率采样，各自适合什么场景？

下一节：**阶段 ① · 读懂 nanoGPT**。